## Project: Analyzing New York Data and Sickness

New York City is one of the most popular and dense cities to live in - it offers many different opportunties, education, and life activities where indiviual can gather and perform.

In this project I am going to be analyzing the weather of New York, focus on four season and the attributes that comes with it.

- Farenheit
- Air Quality
- Precipitation
- Wind Direction
- Sunset
- Humditiy
- Visiblity
- Pressure

Also we are going to be using people occupation and sickess as well.

This will give us a much more deeper story that will be provided in data and give us a closer analysis.



In [81]:
import pandas as pd

import duckdb
import numpy as np
import geopandas
from meteostat import Point, daily
import meteostat as ms
from datetime import datetime
from epiweeks import Week


- temp : Avg Temp °C
- tmin/tmax Daily low / high °C
- rhum: Relative humidity %
- prcp: recipitation mm
- snwd: Snow Depth cm
- wspd: Avg wind speed km/h
- pres Sea Level Air pressure hpa
- tsun Sunshine duration minutes
- clddc cloud cover oktas(0 = clear, 8 = overcast )

In [82]:
start = datetime (2020, 1, 1)
end = datetime (2025, 12, 31)

jfk_data = daily('74486', start, end) # This is the meteostat & WMO of jfk
jfk_df = jfk_data.fetch(units=ms.UnitSystem.IMPERIAL)

ny_point = Point(40.7128, -74.0060)

stations = ms.stations.nearby(ny_point, limit=4)
ts = ms.daily(stations, start, end)
ny_df = ms.interpolate(ts, ny_point).fetch(units=ms.UnitSystem.IMPERIAL)

ny_df.describe()
jfk_df.describe()

pd.set_option('display.max_rows', None)

print(ny_df)
print(jfk_df)

print(len(jfk_df))
print(len(ny_df))

# This is different from 

print(stations.head())

ny_df.to_csv('ny_2025.csv')



            temp  tmin   tmax  rhum   prcp    snwd  wspd  wpgt    pres  tsun  \
time                                                                           
2020-01-01  38.5  35.1   41.0    57    0.0     0.0  10.7  <NA>  1008.2  <NA>   
2020-01-02  40.5  33.1   48.0    61    0.0     0.0   7.7  <NA>  1013.9  <NA>   
2020-01-03  45.7  44.1   46.9    84  0.126     0.0   5.2  <NA>  1010.2  <NA>   
2020-01-04  46.8  44.1   48.9    94  0.224     0.0   3.5  <NA>  1003.7  <NA>   
2020-01-05  40.3  37.0   45.0    53    0.0     0.0   5.1  <NA>  1010.1  <NA>   
2020-01-06  39.0  31.6   45.0    68  0.024   1.181   9.9  <NA>  1015.3  <NA>   
2020-01-07  40.5  37.9   44.1    57    0.0     0.0   6.5  <NA>  1016.2  <NA>   
2020-01-08  36.7  30.9   44.1    62    0.0     0.0   8.3  <NA>  1016.6  <NA>   
2020-01-09  30.9  26.1   36.0    48    0.0     0.0   5.7  <NA>  1038.8  <NA>   
2020-01-10  43.5  34.0   53.1    72    0.0     0.0   9.0  <NA>  1034.5  <NA>   
2020-01-11  58.5  48.0   67.8    73    0

In [83]:
ny = pd.read_csv('ny_2025.csv', index_col = 'time', parse_dates=True)
print(ny.info())

ny = ny.drop(columns=['wpgt', 'tsun'])

<class 'pandas.DataFrame'>
DatetimeIndex: 2192 entries, 2020-01-01 to 2025-12-31
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   temp    2192 non-null   float64
 1   tmin    2192 non-null   float64
 2   tmax    2192 non-null   float64
 3   rhum    2192 non-null   int64  
 4   prcp    2192 non-null   float64
 5   snwd    2190 non-null   float64
 6   wspd    2192 non-null   float64
 7   wpgt    4 non-null      float64
 8   pres    2192 non-null   float64
 9   tsun    0 non-null      float64
 10  cldc    1540 non-null   float64
dtypes: float64(10), int64(1)
memory usage: 205.5 KB
None


In [84]:
full_range = pd.date_range(ny.index.min(), ny.index.max(), freq = 'D')

missing_days = full_range.difference(ny.index)
print(len(missing_days), missing_days[:10])

ny = ny.reindex(full_range)

bad_temp = ny[(ny['tmin'] > ny['temp']) | (ny['temp'] > ny['tmax'])]
print(bad_temp)

print(ny[ny['prcp']< 0])
print(ny[ny['rhum'] > 100])

0 DatetimeIndex([], dtype='datetime64[us]', freq='D')
Empty DataFrame
Columns: [temp, tmin, tmax, rhum, prcp, snwd, wspd, pres, cldc]
Index: []
Empty DataFrame
Columns: [temp, tmin, tmax, rhum, prcp, snwd, wspd, pres, cldc]
Index: []
Empty DataFrame
Columns: [temp, tmin, tmax, rhum, prcp, snwd, wspd, pres, cldc]
Index: []


In [85]:
ny.loc[ny['rhum']> 100, 'rhum'] = np.nan

smooth_cols = ['temp', 'tmin', 'tmax', 'rhum', 'pres', 'wspd']
ny[smooth_cols] = ny[smooth_cols].interpolate(limit=3)

ny['snwd'] = ny['snwd'].fillna(0)

ny.to_csv('ny_2020_2025_clean.csv')

ny.index.name = 'time'

monthly = ny['temp'].resample('ME').mean().reset_index()
monthly['Year'] = monthly['time'].dt.year
monthly['Month'] = monthly['time'].dt.month_name()

print(monthly.head(100))
print(len(monthly))
print(monthly['temp'].isna().sum())

         time       temp  Year      Month
0  2020-01-31  40.300000  2020    January
1  2020-02-29  40.620690  2020   February
2  2020-03-31  47.841935  2020      March
3  2020-04-30  50.656667  2020      April
4  2020-05-31  60.454839  2020        May
5  2020-06-30  73.403333  2020       June
6  2020-07-31  80.406452  2020       July
7  2020-08-31  78.329032  2020     August
8  2020-09-30  71.070000  2020  September
9  2020-10-31  60.135484  2020    October
10 2020-11-30  53.840000  2020   November
11 2020-12-31  40.851613  2020   December
12 2021-01-31  36.500000  2021    January
13 2021-02-28  35.378571  2021   February
14 2021-03-31  45.861290  2021      March
15 2021-04-30  53.930000  2021      April
16 2021-05-31  62.574194  2021        May
17 2021-06-30  73.820000  2021       June
18 2021-07-31  77.048387  2021       July
19 2021-08-31  78.083871  2021     August
20 2021-09-30  72.683333  2021  September
21 2021-10-31  64.490323  2021    October
22 2021-11-30  47.503333  2021   N

In [86]:
df1_2020_2021 = pd.read_csv('CBSAData_2020-2021.csv')
df2_2021_2022 = pd.read_csv('CBSAData_2021-2022.csv')
df3_2022_2023 = pd.read_csv('CBSAData_2022-2023.csv')
df4_2023_2024 = pd.read_csv('CBSAData_2023-2024.csv')
df5_2024_2025 = pd.read_csv('CBSAData_2024-2025.csv')

merged_df = pd.concat([df1_2020_2021, df2_2021_2022, df3_2022_2023, df4_2023_2024, df5_2024_2025], ignore_index=True)

merged_df = merged_df.drop(columns=['WEEKEND'])

print(merged_df.isna().sum())

merged_df.to_csv('Illness_Data_2020_2025.csv', index=False)

print(merged_df.columns.tolist())
print(merged_df['CBSA NAME'].unique())

ill_new_york = merged_df[merged_df['CBSA NAME'] == 'New York-Newark-Jersey City, NY-NJ-PA']

print(len(ill_new_york))
print(ill_new_york['CBSA NAME'].unique())
print(ill_new_york.head())

ill_new_york.to_csv('ill_new_york.csv', index=False)

CBSA_ID                 0
CBSA NAME               0
CBSA                    0
ACTIVITY LEVEL          0
ACTIVITY LEVEL LABEL    0
WEEK                    0
SEASON                  0
dtype: int64
['CBSA_ID', 'CBSA NAME', 'CBSA', 'ACTIVITY LEVEL', 'ACTIVITY LEVEL LABEL', 'WEEK', 'SEASON']
<StringArray>
[               'Aberdeen, SD',                'Aberdeen, WA',
                     'Ada, OK',              'Alamogordo, NM',
                  'Albany, OR', 'Albany-Schenectady-Troy, NY',
               'Albemarle, NC',              'Albert Lea, MN',
             'Albertville, AL',             'Albuquerque, NM',
 ...
                'Marshall, MN',          'Middlesborough, KY',
                 'New Ulm, MN',              'Oak Harbor, WA',
                'Portales, NM',              'Portsmouth, OH',
               'Sandpoint, ID',                'Van Wert, OH',
             'Weatherford, OK',              'Zanesville, OH']
Length: 751, dtype: str
5
<StringArray>
['New York-Newark-Jerse

week = ili_ny['WEEK'].astype(int)
jan4 = pd.to_datetime(ili_ny['YEAR'].astype(str) + '-01-04')
days_to_sat = (5 - jan4.dt.day_of_week) % 7
week1_end = jan4 + pd.to_timedelta(days_to_sat, unit ='D')
ili_ny['week_end'] = week1_end + pd.to_timedelta((week - 1) * 7, unit='D')

In [ ]:
ili = pd.read_csv('ILINet.csv', skiprows=1, na_values=['X'])
clin = pd.read_csv('ICL_NREVSS_Clinical_Labs.csv', skiprows=1, na_values=['X'])

print(ili.columns.to_list())
print(ili['REGION'].unique())


ili_ny = ili[ili['REGION'] == 'New York City'].copy()
clin_nyc = clin[clin['REGION'] == 'New York City'].copy()

print(len(ili_ny), len(clin_nyc))


week = ili_ny['WEEK'].astype(int)
jan4 = pd.to_datetime(ili_ny['YEAR'].astype(str) + '-01-04')
days_to_sat = (5 - jan4.dt.day_of_week) % 7
week1_end = jan4 + pd.to_timedelta(days_to_sat, unit ='D')
ili_ny['week_end'] = week1_end + pd.to_timedelta((week - 1) * 7, unit='D')

# will resume later on this work = clin_nyc[]

['REGION TYPE', 'REGION', 'YEAR', 'WEEK', '% WEIGHTED ILI', '%UNWEIGHTED ILI', 'AGE 0-4', 'AGE 25-49', 'AGE 25-64', 'AGE 5-24', 'AGE 50-64', 'AGE 65', 'ILITOTAL', 'NUM. OF PROVIDERS', 'TOTAL PATIENTS']
<StringArray>
['New York']
Length: 1, dtype: str
-------------------------------------
0 0
-------------------------------------
